In [1]:
# Keep Kaggle's compiled numpy/pandas/scipy/sklearn stack intact.
# Force-reinstalling those packages can create pandas/numpy ABI errors.
!pip install -q --no-cache-dir \
  "lightgbm==4.5.0" \
  "imbalanced-learn==0.12.4" \
  "python-dotenv" \
  "dateparser" \
  "uniplot"

!pip install -q --no-cache-dir --no-deps git+https://github.com/BorgwardtLab/maldi-learn.git
!pip install -q --no-cache-dir --no-deps git+https://github.com/BorgwardtLab/libTLDA.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 260.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 240.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 167.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 169.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 168.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 157.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 311.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 349.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 300.7 MB/s eta 0:00:00
 

In [ ]:
# Environment sanity check. If this cell fails with a numpy/pandas ABI error,
# restart the Kaggle session and rerun the patched install cell above.
import numpy as np
import pandas as pd
import scipy
import sklearn
import lightgbm
print("numpy", np.__version__, np.__file__)
print("pandas", pd.__version__, pd.__file__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)
print("lightgbm", lightgbm.__version__)


In [2]:
%%writefile /kaggle/working/patch_maldi_learn_topf.py
from pathlib import Path
import site

candidate_roots = site.getsitepackages() + [site.getusersitepackages()]
patched = []

for root in candidate_roots:
    path = Path(root) / "maldi_learn" / "preprocessing" / "__init__.py"
    if not path.exists():
        continue

    text = path.read_text()
    new_text = text.replace("from .topological import TopologicalPeakFiltering\n", "")
    new_text = new_text.replace("from .topological import TopologicalPeakFiltering", "")

    if new_text != text:
        path.write_text(new_text)
        patched.append(str(path))

print("patched:", patched if patched else "nothing changed")


Overwriting /kaggle/working/patch_maldi_learn_topf.py


In [3]:
!python /kaggle/working/patch_maldi_learn_topf.py


patched: nothing changed


In [4]:
%%writefile /kaggle/working/import_check.py
import numpy as np
import pandas as pd
import scipy
import sklearn

print("numpy", np.__version__, np.__file__)
print("pandas", pd.__version__, pd.__file__)
print("scipy", scipy.__version__)
print("sklearn", sklearn.__version__)

from maldi_learn.driams import load_driams_dataset
from maldi_learn.utilities import case_based_stratification, stratify_by_species_and_label
print("maldi-learn import OK")


Overwriting /kaggle/working/import_check.py


In [5]:
!python /kaggle/working/import_check.py


numpy 1.26.4 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py
pandas 2.2.3 /usr/local/lib/python3.12/dist-packages/pandas/__init__.py
scipy 1.12.0
sklearn 1.4.2
maldi-learn import OK


In [6]:
![ -d /kaggle/working/maldi_amr ] || git clone https://github.com/BorgwardtLab/maldi_amr.git /kaggle/working/maldi_amr


In [7]:
%%writefile /kaggle/working/run_background_audit_framework.py
# paste entire run_background_audit_framework.py here
#!/usr/bin/env python3
"""Model-agnostic Background-Matched Transfer Audit.

This script takes prediction rows from any MALDI-AMR model and asks whether the
prediction for a focal antibiotic survives after matching isolates on their
co-resistance background. It is intentionally independent of Mega_Model, Torch,
LightGBM, pandas, and DRIAMS-specific paths.

Minimum input is a long CSV with one row per isolate/drug prediction:

    isolate_id,site,year,organism,drug,label,prob

Column names are configurable so outputs from Weis-style scripts or other
models can be adapted without changing code.
"""

from __future__ import annotations

import argparse
import csv
import math
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, Iterable, List, Sequence, Tuple


DEFAULT_BOOTSTRAP_N = 500
DEFAULT_PERMUTATION_N = 500
DEFAULT_RANDOM_SEED = 20260427

LABEL_CHAR = {0: "S", 1: "R"}
UNKNOWN_CHAR = "U"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Run a model-agnostic background-matched MALDI-AMR audit."
    )
    parser.add_argument("--predictions-csv", required=True)
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--id-col", default="isolate_id")
    parser.add_argument("--site-col", default="site")
    parser.add_argument("--year-col", default="year")
    parser.add_argument("--organism-col", default="organism")
    parser.add_argument("--drug-col", default="drug")
    parser.add_argument("--label-col", default="label")
    parser.add_argument("--prob-col", default="prob")
    parser.add_argument(
        "--background-signature-col",
        default="",
        help="Optional column containing a precomputed non-focal resistance background signature.",
    )
    parser.add_argument("--model-name", default="")
    parser.add_argument(
        "--background-drugs",
        default="",
        help="Optional comma-separated drug list to use in background signatures.",
    )
    parser.add_argument("--min-pos-per-stratum", type=int, default=3)
    parser.add_argument("--min-neg-per-stratum", type=int, default=3)
    parser.add_argument("--match-year", action="store_true")
    parser.add_argument("--bootstrap-n", type=int, default=DEFAULT_BOOTSTRAP_N)
    parser.add_argument("--permutation-n", type=int, default=DEFAULT_PERMUTATION_N)
    parser.add_argument("--random-seed", type=int, default=DEFAULT_RANDOM_SEED)
    parser.add_argument("--adequacy-min-n-matched", type=int, default=100)
    parser.add_argument("--adequacy-min-retention", type=float, default=0.10)
    parser.add_argument("--adequacy-min-pairwise", type=int, default=100)
    parser.add_argument(
        "--sensitivity-thresholds",
        default="2,3,5",
        help="Comma-separated min pos/neg stratum thresholds for sensitivity.",
    )
    parser.add_argument("--min-edge-n", type=int, default=30)
    return parser.parse_args()


def normalize_label(value) -> int | None:
    if value is None:
        return None
    text = str(value).strip()
    if text == "":
        return None
    upper = text.upper()
    if upper in {"R", "RESISTANT", "TRUE", "T", "YES", "Y"}:
        return 1
    if upper in {"S", "SUSCEPTIBLE", "FALSE", "F", "NO", "N"}:
        return 0
    if upper in {"I", "INTERMEDIATE", "U", "UNKNOWN", "NA", "NAN", "NONE", "-", "."}:
        return None
    try:
        number = float(text)
    except ValueError:
        return None
    if math.isnan(number):
        return None
    if number == 1:
        return 1
    if number == 0:
        return 0
    return None


def parse_probability(value) -> float | None:
    try:
        prob = float(str(value).strip())
    except (TypeError, ValueError):
        return None
    if not math.isfinite(prob):
        return None
    return prob


def require_columns(fieldnames: Sequence[str], required: Sequence[str], path: Path | None = None) -> None:
    missing = [name for name in required if name not in fieldnames]
    if missing:
        prefix = f"{path}: " if path else ""
        raise ValueError(f"{prefix}missing required columns: {', '.join(missing)}")


def read_rows_from_records(
    records: Iterable[dict],
    *,
    id_col: str,
    site_col: str,
    year_col: str,
    organism_col: str,
    drug_col: str,
    label_col: str,
    prob_col: str,
    model_name: str = "",
    background_signature_col: str = "",
) -> List[dict]:
    rows: List[dict] = []
    for record in records:
        label = normalize_label(record.get(label_col))
        prob = parse_probability(record.get(prob_col))
        if label is None or prob is None:
            continue
        row = {
            "uid": str(record.get(id_col, "")).strip(),
            "site": str(record.get(site_col, "")).strip(),
            "year": str(record.get(year_col, "")).strip(),
            "organism": str(record.get(organism_col, "")).strip(),
            "drug": str(record.get(drug_col, "")).strip(),
            "label": label,
            "prob": prob,
            "model_name": str(model_name or record.get("model_name", "")).strip(),
        }
        if background_signature_col:
            row["background_signature"] = str(record.get(background_signature_col, "")).strip()
        rows.append(row)
    if not rows:
        raise ValueError("No usable prediction rows after label/probability normalization.")
    return rows


def read_long_predictions(
    path: Path | str,
    *,
    id_col: str,
    site_col: str,
    year_col: str,
    organism_col: str,
    drug_col: str,
    label_col: str,
    prob_col: str,
    model_name: str = "",
    background_signature_col: str = "",
) -> List[dict]:
    path = Path(path)
    with path.open(newline="") as f:
        reader = csv.DictReader(f)
        require_columns(
            reader.fieldnames or [],
            [id_col, site_col, year_col, organism_col, drug_col, label_col, prob_col],
            path,
        )
        if background_signature_col:
            require_columns(reader.fieldnames or [], [background_signature_col], path)
        return read_rows_from_records(
            reader,
            id_col=id_col,
            site_col=site_col,
            year_col=year_col,
            organism_col=organism_col,
            drug_col=drug_col,
            label_col=label_col,
            prob_col=prob_col,
            model_name=model_name,
            background_signature_col=background_signature_col,
        )


def first_seen_drugs(rows: Sequence[dict]) -> List[str]:
    seen = set()
    drugs = []
    for row in rows:
        drug = row["drug"]
        if drug not in seen:
            seen.add(drug)
            drugs.append(drug)
    return drugs


def isolate_key(row: dict) -> Tuple[str, str, str, str]:
    return (row["site"], row["year"], row["organism"], row["uid"])


def label_matrix(rows: Sequence[dict]) -> Dict[Tuple[str, str, str, str], Dict[str, int]]:
    matrix: Dict[Tuple[str, str, str, str], Dict[str, int]] = defaultdict(dict)
    for row in rows:
        matrix[isolate_key(row)][row["drug"]] = int(row["label"])
    return matrix


def add_background_signatures(rows: Sequence[dict], background_drugs: Sequence[str] | None = None) -> List[dict]:
    # Prefer exporter-provided signatures. This matters for Weis-style external validation
    # because the focal-drug prediction subsets can differ by drug, while the exporter can
    # build the co-resistance background from the full AST table by isolate ID.
    if rows and all(str(row.get("background_signature", "")).strip() for row in rows):
        enriched = []
        for row in rows:
            signature = str(row.get("background_signature", "")).strip()
            known = 0
            resistant = 0
            for part in signature.split("|"):
                if "=" not in part:
                    continue
                value = part.rsplit("=", 1)[-1]
                if value in {"S", "R"}:
                    known += 1
                    resistant += int(value == "R")
            new_row = dict(row)
            new_row["background_signature"] = signature
            new_row["background_known_count"] = known
            new_row["background_resistant_count"] = resistant
            enriched.append(new_row)
        return enriched

    drugs = list(background_drugs) if background_drugs else first_seen_drugs(rows)
    matrix = label_matrix(rows)
    enriched = []
    for row in rows:
        labels = matrix[isolate_key(row)]
        parts = []
        known = 0
        resistant = 0
        for drug in drugs:
            if drug == row["drug"]:
                continue
            label = labels.get(drug)
            char = LABEL_CHAR.get(label, UNKNOWN_CHAR)
            parts.append(f"{drug}={char}")
            if label in (0, 1):
                known += 1
                resistant += int(label == 1)
        new_row = dict(row)
        new_row["background_signature"] = "|".join(parts) if parts else "NO_BACKGROUND_DRUGS"
        new_row["background_known_count"] = known
        new_row["background_resistant_count"] = resistant
        enriched.append(new_row)
    return enriched

def safe_auc(labels: Sequence[int], scores: Sequence[float]) -> float:
    pairs = [(int(y), float(s)) for y, s in zip(labels, scores) if math.isfinite(float(s))]
    if not pairs:
        return math.nan
    labels = [p[0] for p in pairs]
    n_pos = sum(labels)
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return math.nan

    order = sorted(range(len(pairs)), key=lambda i: pairs[i][1])
    ranks = [0.0] * len(pairs)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and pairs[order[j + 1]][1] == pairs[order[i]][1]:
            j += 1
        avg_rank = 0.5 * (i + j) + 1.0
        for k in range(i, j + 1):
            ranks[order[k]] = avg_rank
        i = j + 1
    pos_rank_sum = sum(rank for rank, label in zip(ranks, labels) if label == 1)
    return (pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)


def safe_aupr(labels: Sequence[int], scores: Sequence[float]) -> float:
    pairs = [(int(y), float(s)) for y, s in zip(labels, scores) if math.isfinite(float(s))]
    n_pos = sum(y for y, _ in pairs)
    if n_pos == 0:
        return math.nan
    pairs.sort(key=lambda p: p[1], reverse=True)
    tp = 0
    precisions = []
    for idx, (label, _) in enumerate(pairs, start=1):
        if label == 1:
            tp += 1
            precisions.append(tp / idx)
    return sum(precisions) / n_pos


def bootstrap_metric_ci(
    labels: Sequence[int],
    scores: Sequence[float],
    metric_fn,
    *,
    n_boot: int,
    seed: int,
    ci: float = 0.95,
) -> Tuple[float, float]:
    if n_boot <= 0:
        return math.nan, math.nan
    pairs = [(int(y), float(s)) for y, s in zip(labels, scores) if math.isfinite(float(s))]
    if not pairs or sum(y for y, _ in pairs) == 0:
        return math.nan, math.nan
    rng = random.Random(seed)
    values = []
    for _ in range(n_boot):
        sample = [pairs[rng.randrange(len(pairs))] for _ in pairs]
        value = metric_fn([y for y, _ in sample], [s for _, s in sample])
        if not math.isnan(value):
            values.append(value)
    if not values:
        return math.nan, math.nan
    values.sort()
    alpha = (1.0 - ci) / 2.0
    low_idx = min(len(values) - 1, max(0, int(math.floor(alpha * len(values)))))
    high_idx = min(len(values) - 1, max(0, int(math.ceil((1.0 - alpha) * len(values))) - 1))
    return values[low_idx], values[high_idx]


def group_by(rows: Iterable[dict], keys: Sequence[str]) -> Dict[Tuple, List[dict]]:
    grouped: Dict[Tuple, List[dict]] = defaultdict(list)
    for row in rows:
        grouped[tuple(row.get(key, "") for key in keys)].append(row)
    return grouped


def pairwise_accuracy_for_strata(rows: Sequence[dict], stratum_cols: Sequence[str], score_col: str = "prob") -> Tuple[float, int]:
    wins = 0.0
    total = 0
    for group in group_by(rows, stratum_cols).values():
        pos = [float(r[score_col]) for r in group if int(r["label"]) == 1]
        neg = [float(r[score_col]) for r in group if int(r["label"]) == 0]
        for p in pos:
            for n in neg:
                if p > n:
                    wins += 1.0
                elif p == n:
                    wins += 0.5
                total += 1
    return (wins / total if total else math.nan), total


def permutation_null_within_strata(
    rows: Sequence[dict],
    stratum_cols: Sequence[str],
    *,
    score_col: str,
    observed: float,
    n_perm: int,
    seed: int,
) -> Tuple[float, float, float]:
    if n_perm <= 0 or not rows or math.isnan(observed):
        return math.nan, math.nan, math.nan
    rng = random.Random(seed)
    groups = list(group_by(rows, stratum_cols).values())
    null_values = []
    for _ in range(n_perm):
        shuffled_rows = []
        for group in groups:
            labels = [int(r["label"]) for r in group]
            rng.shuffle(labels)
            for row, label in zip(group, labels):
                new_row = dict(row)
                new_row["label"] = label
                shuffled_rows.append(new_row)
        value = safe_auc(
            [int(r["label"]) for r in shuffled_rows],
            [float(r[score_col]) for r in shuffled_rows],
        )
        if not math.isnan(value):
            null_values.append(value)
    if not null_values:
        return math.nan, math.nan, math.nan
    p_value = (1.0 + sum(value >= observed for value in null_values)) / (1.0 + len(null_values))
    mean = sum(null_values) / len(null_values)
    if len(null_values) > 1:
        var = sum((value - mean) ** 2 for value in null_values) / (len(null_values) - 1)
        std = math.sqrt(var)
    else:
        std = 0.0
    return p_value, mean, std


def assign_adequacy_label(
    *,
    n_matched: int,
    matched_retention: float,
    n_valid_strata: int,
    pairwise_comparisons: int,
    min_n_matched: int,
    min_retention: float,
    min_pairwise: int,
) -> str:
    if n_matched == 0 or n_valid_strata == 0:
        return "not_interpretable_no_valid_strata"
    issues = []
    if n_matched < min_n_matched:
        issues.append("low_n_matched")
    if matched_retention < min_retention:
        issues.append("low_retention")
    if pairwise_comparisons < min_pairwise:
        issues.append("low_pairwise")
    if issues:
        return "caution_" + "_and_".join(issues)
    return "interpretable"


def interpretation_category(row: dict) -> str:
    adequacy = str(row.get("adequacy_label", ""))
    raw_auc = float(row.get("raw_auc", math.nan))
    centered = float(row.get("stratum_centered_auc", math.nan))
    if adequacy.startswith("not_interpretable"):
        return "insufficient_matched_overlap"
    if adequacy.startswith("caution"):
        return "caution_low_matched_support"
    if math.isnan(raw_auc) or raw_auc < 0.60:
        return "weak_raw_signal"
    if math.isnan(centered):
        return "insufficient_matched_overlap"
    if centered >= 0.60:
        return "focal_signal_retained"
    if centered >= 0.55:
        return "partially_retained_or_uncertain"
    return "background_driven_collapse"


def compute_background_matched_summary(
    rows: Sequence[dict],
    *,
    min_pos_per_stratum: int = 3,
    min_neg_per_stratum: int = 3,
    match_year: bool = False,
    bootstrap_n: int = DEFAULT_BOOTSTRAP_N,
    permutation_n: int = DEFAULT_PERMUTATION_N,
    random_seed: int = DEFAULT_RANDOM_SEED,
    adequacy_min_n_matched: int = 100,
    adequacy_min_retention: float = 0.10,
    adequacy_min_pairwise: int = 100,
) -> Tuple[List[dict], List[dict]]:
    summary = []
    retained_rows = []
    stratum_cols = ["background_signature"] + (["year"] if match_year else [])
    pair_groups = group_by(rows, ["site", "organism", "drug"])

    for group_idx, ((site, organism, drug), pair_rows) in enumerate(sorted(pair_groups.items())):
        labels = [int(r["label"]) for r in pair_rows]
        probs = [float(r["prob"]) for r in pair_rows]
        raw_auc = safe_auc(labels, probs)
        raw_aupr = safe_aupr(labels, probs)
        seed_base = int(random_seed) + group_idx * 100003
        raw_low, raw_high = bootstrap_metric_ci(labels, probs, safe_auc, n_boot=bootstrap_n, seed=seed_base + 1)

        matched = []
        for stratum_rows in group_by(pair_rows, stratum_cols).values():
            n_pos = sum(int(r["label"]) == 1 for r in stratum_rows)
            n_neg = sum(int(r["label"]) == 0 for r in stratum_rows)
            if n_pos >= min_pos_per_stratum and n_neg >= min_neg_per_stratum:
                matched.extend(stratum_rows)

        if matched:
            centered = []
            for stratum_rows in group_by(matched, stratum_cols).values():
                mean_prob = sum(float(r["prob"]) for r in stratum_rows) / len(stratum_rows)
                for row in stratum_rows:
                    new_row = dict(row)
                    new_row["centered_prob"] = float(row["prob"]) - mean_prob
                    new_row["matched_valid_stratum"] = True
                    centered.append(new_row)
            matched = centered
            matched_labels = [int(r["label"]) for r in matched]
            matched_probs = [float(r["prob"]) for r in matched]
            centered_probs = [float(r["centered_prob"]) for r in matched]
            matched_auc = safe_auc(matched_labels, matched_probs)
            matched_aupr = safe_aupr(matched_labels, matched_probs)
            centered_auc = safe_auc(matched_labels, centered_probs)
            matched_low, matched_high = bootstrap_metric_ci(
                matched_labels, matched_probs, safe_auc, n_boot=bootstrap_n, seed=seed_base + 2
            )
            centered_low, centered_high = bootstrap_metric_ci(
                matched_labels, centered_probs, safe_auc, n_boot=bootstrap_n, seed=seed_base + 3
            )
            permutation_p, null_mean, null_std = permutation_null_within_strata(
                matched,
                stratum_cols,
                score_col="centered_prob",
                observed=centered_auc,
                n_perm=permutation_n,
                seed=seed_base + 4,
            )
            pair_acc, pair_n = pairwise_accuracy_for_strata(matched, stratum_cols)
            matched_retention = len(matched) / len(pair_rows)
            n_valid_strata = len(group_by(matched, stratum_cols))
            retained_rows.extend(matched)
        else:
            matched_auc = matched_aupr = centered_auc = math.nan
            matched_low = matched_high = centered_low = centered_high = math.nan
            permutation_p = null_mean = null_std = math.nan
            pair_acc = math.nan
            pair_n = 0
            matched_retention = 0.0
            n_valid_strata = 0

        adequacy = assign_adequacy_label(
            n_matched=len(matched),
            matched_retention=matched_retention,
            n_valid_strata=n_valid_strata,
            pairwise_comparisons=pair_n,
            min_n_matched=adequacy_min_n_matched,
            min_retention=adequacy_min_retention,
            min_pairwise=adequacy_min_pairwise,
        )
        row = {
            "site": site,
            "organism": organism,
            "drug": drug,
            "raw_auc": raw_auc,
            "raw_auc_ci_low": raw_low,
            "raw_auc_ci_high": raw_high,
            "raw_aupr": raw_aupr,
            "matched_auc": matched_auc,
            "matched_auc_ci_low": matched_low,
            "matched_auc_ci_high": matched_high,
            "matched_aupr": matched_aupr,
            "stratum_centered_auc": centered_auc,
            "stratum_centered_auc_ci_low": centered_low,
            "stratum_centered_auc_ci_high": centered_high,
            "pairwise_accuracy": pair_acc,
            "pairwise_comparisons": pair_n,
            "permutation_p": permutation_p,
            "permutation_null_mean": null_mean,
            "permutation_null_std": null_std,
            "matched_retention": matched_retention,
            "adequacy_label": adequacy,
            "interpretation_category": "",
            "n_total": len(pair_rows),
            "n_r": sum(labels),
            "n_matched": len(matched),
            "n_matched_r": sum(int(r["label"]) for r in matched) if matched else 0,
            "n_valid_strata": n_valid_strata,
            "min_pos_per_stratum": min_pos_per_stratum,
            "min_neg_per_stratum": min_neg_per_stratum,
            "match_year": bool(match_year),
        }
        row["interpretation_category"] = interpretation_category(row)
        summary.append(row)
    return summary, retained_rows


def parse_thresholds(text: str) -> List[int]:
    values = []
    for part in str(text or "").split(","):
        part = part.strip()
        if not part:
            continue
        value = int(part)
        if value <= 0:
            raise ValueError("Sensitivity thresholds must be positive.")
        values.append(value)
    return sorted(set(values))


def phi_from_counts(n00: int, n01: int, n10: int, n11: int) -> float:
    denom = math.sqrt((n11 + n10) * (n01 + n00) * (n11 + n01) * (n10 + n00))
    if denom == 0:
        return math.nan
    return (n11 * n00 - n10 * n01) / denom


def build_isolate_label_records(rows: Sequence[dict]) -> List[dict]:
    by_iso: Dict[Tuple[str, str, str, str], dict] = {}
    for row in rows:
        key = isolate_key(row)
        record = by_iso.setdefault(
            key,
            {"site": row["site"], "year": row["year"], "organism": row["organism"], "uid": row["uid"]},
        )
        record[row["drug"]] = int(row["label"])
    return list(by_iso.values())


def pair_metrics(records: Sequence[dict], drug_a: str, drug_b: str) -> dict | None:
    n00 = n01 = n10 = n11 = 0
    for record in records:
        if drug_a not in record or drug_b not in record:
            continue
        a = int(record[drug_a])
        b = int(record[drug_b])
        if a == 0 and b == 0:
            n00 += 1
        elif a == 0 and b == 1:
            n01 += 1
        elif a == 1 and b == 0:
            n10 += 1
        elif a == 1 and b == 1:
            n11 += 1
    n = n00 + n01 + n10 + n11
    if n == 0:
        return None
    prev_a = (n10 + n11) / n
    prev_b = (n01 + n11) / n
    expected_rr = prev_a * prev_b
    observed_rr = n11 / n
    lift = observed_rr / expected_rr if expected_rr > 0 else math.nan
    return {
        "drug_a": drug_a,
        "drug_b": drug_b,
        "n_both_known": n,
        "n_ss": n00,
        "n_sr": n01,
        "n_rs": n10,
        "n_rr": n11,
        "prevalence_a": prev_a,
        "prevalence_b": prev_b,
        "p_rr_observed": observed_rr,
        "p_rr_expected_independent": expected_rr,
        "rr_lift": lift,
        "phi": phi_from_counts(n00, n01, n10, n11),
        "resistant_jaccard": n11 / (n11 + n10 + n01) if (n11 + n10 + n01) else math.nan,
    }


def build_cross_resistance_edges(rows: Sequence[dict], min_edge_n: int) -> Tuple[List[dict], List[dict]]:
    records = build_isolate_label_records(rows)
    drugs = first_seen_drugs(rows)
    by_site: Dict[str, List[dict]] = defaultdict(list)
    by_site["ALL"] = list(records)
    for record in records:
        by_site[record["site"]].append(record)

    edges = []
    prevalence = []
    for site, site_records in sorted(by_site.items()):
        for drug in drugs:
            vals = [record[drug] for record in site_records if drug in record]
            if vals:
                prevalence.append(
                    {
                        "site": site,
                        "drug": drug,
                        "n_known": len(vals),
                        "n_resistant": sum(vals),
                        "prevalence": sum(vals) / len(vals),
                    }
                )
        for i, drug_a in enumerate(drugs):
            for drug_b in drugs[i + 1 :]:
                metrics = pair_metrics(site_records, drug_a, drug_b)
                if metrics and metrics["n_both_known"] >= min_edge_n:
                    edges.append({"site": site, **metrics})
    return edges, prevalence


def strongest_partner_by_drug(edges: Sequence[dict]) -> Dict[str, dict]:
    partners = {}
    for edge in edges:
        if edge.get("site") != "ALL":
            continue
        for drug, other in [(edge["drug_a"], edge["drug_b"]), (edge["drug_b"], edge["drug_a"])]:
            current = partners.get(drug)
            if current is None or float(edge["phi"]) > float(current["phi"]):
                partners[drug] = {"partner": other, "phi": edge["phi"], "rr_lift": edge["rr_lift"]}
    return partners


def add_ecology_annotations(summary: Sequence[dict], edges: Sequence[dict]) -> List[dict]:
    partners = strongest_partner_by_drug(edges)
    annotated = []
    for row in summary:
        partner = partners.get(row["drug"], {})
        new_row = dict(row)
        new_row["strongest_network_partner"] = partner.get("partner", "")
        new_row["partner_phi"] = partner.get("phi", math.nan)
        new_row["partner_rr_lift"] = partner.get("rr_lift", math.nan)
        annotated.append(new_row)
    return annotated


def safe_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        if math.isnan(value):
            return ""
        return f"{value:.6g}"
    return str(value)


def write_csv(path: Path, rows: Sequence[dict], fieldnames: Sequence[str] | None = None) -> None:
    if fieldnames is None:
        fields = []
        seen = set()
        for row in rows:
            for key in row:
                if key not in seen:
                    seen.add(key)
                    fields.append(key)
    else:
        fields = list(fieldnames)
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({field: safe_text(row.get(field)) for field in fields})


def write_markdown_table(path: Path, title: str, rows: Sequence[dict], fields: Sequence[str]) -> None:
    lines = [f"# {title}", ""]
    if not rows:
        lines.append("No rows.")
    else:
        lines.append("| " + " | ".join(fields) + " |")
        lines.append("| " + " | ".join(["---"] * len(fields)) + " |")
        for row in rows:
            lines.append("| " + " | ".join(safe_text(row.get(field)) for field in fields) + " |")
    path.write_text("\n".join(lines) + "\n")


def write_report(path: Path, summary: Sequence[dict], edges: Sequence[dict]) -> None:
    top_edges = sorted(
        [edge for edge in edges if edge["site"] == "ALL"],
        key=lambda edge: float(edge["phi"]),
        reverse=True,
    )[:10]
    lines = [
        "# Background-Matched Transfer Audit Report",
        "",
        "## What This Tests",
        "",
        "The audit asks whether focal-drug prediction survives after matching isolates by co-resistance background.",
        "Raw AUC can be high because a model learned resistant-population background; stratum-centered AUC is the stricter within-background test.",
        "",
        "## Strongest Co-Resistance Edges",
        "",
        "| Drug A | Drug B | Phi | Lift | n RR | n |",
        "| --- | --- | ---: | ---: | ---: | ---: |",
    ]
    for edge in top_edges:
        lines.append(
            f"| {edge['drug_a']} | {edge['drug_b']} | {safe_text(edge['phi'])} | "
            f"{safe_text(edge['rr_lift'])} | {edge['n_rr']} | {edge['n_both_known']} |"
        )
    lines += [
        "",
        "## Audit Summary",
        "",
        "| Site | Organism | Drug | Raw AUC | Centered AUC | Retention | Adequacy | Interpretation |",
        "| --- | --- | --- | ---: | ---: | ---: | --- | --- |",
    ]
    for row in summary:
        lines.append(
            f"| {row['site']} | {row['organism']} | {row['drug']} | {safe_text(row['raw_auc'])} | "
            f"{safe_text(row['stratum_centered_auc'])} | {safe_text(row['matched_retention'])} | "
            f"{row['adequacy_label']} | {row['interpretation_category']} |"
        )
    path.write_text("\n".join(lines) + "\n")


def write_minimal_network_svg(path: Path, edges: Sequence[dict]) -> None:
    all_edges = [edge for edge in edges if edge["site"] == "ALL"]
    drugs = []
    for edge in all_edges:
        for drug in (edge["drug_a"], edge["drug_b"]):
            if drug not in drugs:
                drugs.append(drug)
    if not drugs:
        path.write_text('<svg xmlns="http://www.w3.org/2000/svg" width="400" height="120"><text x="20" y="40">No network edges</text></svg>')
        return
    width = 760
    height = 520
    cx = width / 2
    cy = height / 2 + 20
    radius = 170
    pos = {}
    for idx, drug in enumerate(drugs):
        angle = -math.pi / 2 + 2 * math.pi * idx / len(drugs)
        pos[drug] = (cx + radius * math.cos(angle), cy + radius * math.sin(angle))
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        '<text x="24" y="34" font-family="Arial" font-size="19" font-weight="700">Cross-resistance network</text>',
    ]
    for edge in all_edges:
        phi = float(edge["phi"])
        if abs(phi) < 0.15:
            continue
        x1, y1 = pos[edge["drug_a"]]
        x2, y2 = pos[edge["drug_b"]]
        color = "#3b5bdb" if phi >= 0 else "#d9480f"
        width_px = 1.0 + 7.0 * min(abs(phi), 1.0)
        parts.append(f'<line x1="{x1}" y1="{y1}" x2="{x2}" y2="{y2}" stroke="{color}" stroke-width="{width_px:.2f}" stroke-opacity="0.58"/>')
    for drug, (x, y) in pos.items():
        label = drug if len(drug) <= 18 else drug[:17] + "..."
        parts.append(f'<circle cx="{x}" cy="{y}" r="30" fill="#f8f9fa" stroke="#222"/>')
        parts.append(f'<text x="{x}" y="{y + 4}" text-anchor="middle" font-family="Arial" font-size="11">{label}</text>')
    parts.append("</svg>")
    path.write_text("\n".join(parts))


def main() -> None:
    args = parse_args()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    background_drugs = [part.strip() for part in args.background_drugs.split(",") if part.strip()]

    rows = read_long_predictions(
        args.predictions_csv,
        id_col=args.id_col,
        site_col=args.site_col,
        year_col=args.year_col,
        organism_col=args.organism_col,
        drug_col=args.drug_col,
        label_col=args.label_col,
        prob_col=args.prob_col,
        model_name=args.model_name,
        background_signature_col=args.background_signature_col,
    )
    rows = add_background_signatures(rows, background_drugs or None)
    summary, retained = compute_background_matched_summary(
        rows,
        min_pos_per_stratum=args.min_pos_per_stratum,
        min_neg_per_stratum=args.min_neg_per_stratum,
        match_year=args.match_year,
        bootstrap_n=args.bootstrap_n,
        permutation_n=args.permutation_n,
        random_seed=args.random_seed,
        adequacy_min_n_matched=args.adequacy_min_n_matched,
        adequacy_min_retention=args.adequacy_min_retention,
        adequacy_min_pairwise=args.adequacy_min_pairwise,
    )
    edges, prevalence = build_cross_resistance_edges(rows, args.min_edge_n)
    annotated = add_ecology_annotations(summary, edges)

    write_csv(output_dir / "normalized_predictions.csv", rows)
    write_csv(output_dir / "background_matched_audit_summary.csv", summary)
    write_csv(output_dir / "background_matched_retained_rows.csv", retained)
    write_csv(output_dir / "cross_resistance_edges.csv", edges)
    write_csv(output_dir / "cross_resistance_prevalence.csv", prevalence)
    write_csv(output_dir / "background_audit_with_resistance_ecology.csv", annotated)
    write_markdown_table(
        output_dir / "background_matched_audit_summary.md",
        "Background-Matched Audit Summary",
        summary,
        [
            "site",
            "organism",
            "drug",
            "raw_auc",
            "matched_auc",
            "stratum_centered_auc",
            "matched_retention",
            "adequacy_label",
            "interpretation_category",
        ],
    )

    sensitivity_rows = []
    for threshold in parse_thresholds(args.sensitivity_thresholds):
        sens_summary, _ = compute_background_matched_summary(
            rows,
            min_pos_per_stratum=threshold,
            min_neg_per_stratum=threshold,
            match_year=args.match_year,
            bootstrap_n=0,
            permutation_n=0,
            random_seed=args.random_seed,
            adequacy_min_n_matched=args.adequacy_min_n_matched,
            adequacy_min_retention=args.adequacy_min_retention,
            adequacy_min_pairwise=args.adequacy_min_pairwise,
        )
        for row in sens_summary:
            sensitivity_rows.append({"sensitivity_threshold": threshold, **row})
    write_csv(output_dir / "background_matched_sensitivity.csv", sensitivity_rows)
    write_report(output_dir / "background_audit_report.md", summary, edges)
    write_minimal_network_svg(output_dir / "cross_resistance_network.svg", edges)

    print(f"Read {len(rows)} normalized prediction rows.")
    print(f"Wrote audit outputs to {output_dir}")
    print(f"Summary rows: {len(summary)}")
    print(f"Cross-resistance edges: {len(edges)}")


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/run_background_audit_framework.py


In [8]:
%%writefile /kaggle/working/export_weis_predictions_for_audit.py
# paste entire export_weis_predictions_for_audit.py here
#!/usr/bin/env python3
"""Export Weis/Borgwardt MALDI-AMR predictions for background-matched auditing.

The Weis repository's stored JSON result files contain scores and labels, but
not isolate identifiers. Background-matched auditing needs isolate identifiers
and co-resistance background labels, so this script reruns the Weis-style
external validation workflow and writes an ID-preserving long prediction table.

Typical Kaggle usage:

    python export_weis_predictions_for_audit.py \
      --weis-repo /kaggle/working/maldi_amr \
      --driams-root /kaggle/input/datasets/drscarlat/driams \
      --species "Escherichia coli" \
      --drugs "Ciprofloxacin,Norfloxacin,Amoxicillin-Clavulanic acid,Ceftriaxone,Ceftazidime,Cefepime" \
      --model lr \
      --seed 35 \
      --output-dir /kaggle/working/weis_background_audit

The script writes:

    weis_predictions_long.csv
    weis_raw_results.json
    audit/background_matched_audit_summary.csv
    audit/background_audit_report.md
"""

from __future__ import annotations

import argparse
import csv
import importlib.util
import json
import math
import os
import pathlib
import subprocess
import sys
from typing import Iterable

import numpy as np


LABEL_CHAR = {0: "S", 1: "R"}
UNKNOWN_CHAR = "U"


def import_from_path(module_name: str, path: pathlib.Path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import {module_name} from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def split_csv_arg(text: str) -> list[str]:
    return [part.strip() for part in str(text or "").split(",") if part.strip()]


def load_weis_modules(weis_repo: pathlib.Path):
    ml_dir = weis_repo / "amr_maldi_ml"
    if not ml_dir.exists():
        raise FileNotFoundError(f"Weis amr_maldi_ml directory not found: {ml_dir}")
    sys.path.insert(0, str(ml_dir))
    models = import_from_path("weis_models_for_audit", ml_dir / "models.py")
    return models


def normalize_ast_value(value) -> int | None:
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    upper = text.upper()
    if upper in {"R", "RESISTANT", "TRUE", "1"}:
        return 1
    if upper in {"S", "SUSCEPTIBLE", "FALSE", "0"}:
        return 0
    try:
        number = float(text)
    except ValueError:
        return None
    if number == 0:
        return 0
    if number == 1:
        return 1
    return None


def code_series_from_dataset(dataset) -> list[str]:
    meta = dataset.y
    for col in ("code", "Code", "sample_id", "id", "uuid"):
        if col in meta.columns:
            return [str(value) for value in meta[col].values]
    return [str(idx) for idx in meta.index]


def load_dataset_for_focal(
    *,
    driams_root: str,
    site: str,
    years: str | Iterable[str],
    species: str,
    antibiotic: str,
):
    from maldi_learn.driams import load_driams_dataset
    from maldi_learn.filters import DRIAMSBooleanExpressionFilter
    from maldi_learn.utilities import case_based_stratification, stratify_by_species_and_label

    extra_filters = []
    if site == "DRIAMS-A":
        extra_filters.append(DRIAMSBooleanExpressionFilter("workstation != HospitalHygiene"))
        id_suffix = "strat"
        strat_fn = case_based_stratification
    else:
        id_suffix = "clean"
        strat_fn = stratify_by_species_and_label

    dataset = load_driams_dataset(
        driams_root,
        site,
        years=years,
        species=species,
        antibiotics=antibiotic,
        handle_missing_resistance_measurements="remove_if_all_missing",
        spectra_type="binned_6000",
        on_error="warn",
        id_suffix=id_suffix,
        extra_filters=extra_filters,
    )
    x = np.asarray([spectrum.intensities for spectrum in dataset.X])
    y = dataset.to_numpy(antibiotic)
    codes = code_series_from_dataset(dataset)
    return dataset, x, y, codes, strat_fn


def load_background_label_map(
    *,
    driams_root: str,
    site: str,
    years: str | Iterable[str],
    species: str,
    drugs: list[str],
) -> dict[str, dict[str, int | None]]:
    from maldi_learn.driams import load_driams_dataset
    from maldi_learn.filters import DRIAMSBooleanExpressionFilter

    extra_filters = []
    id_suffix = "clean"
    if site == "DRIAMS-A":
        extra_filters.append(DRIAMSBooleanExpressionFilter("workstation != HospitalHygiene"))
        id_suffix = "strat"

    dataset = load_driams_dataset(
        driams_root,
        site,
        years=years,
        species=species,
        antibiotics=drugs,
        handle_missing_resistance_measurements="remove_if_all_missing",
        spectra_type="binned_6000",
        on_error="warn",
        id_suffix=id_suffix,
        extra_filters=extra_filters,
    )
    codes = code_series_from_dataset(dataset)
    by_code: dict[str, dict[str, int | None]] = {}
    for idx, code in enumerate(codes):
        labels: dict[str, int | None] = {}
        for drug in drugs:
            if drug in dataset.y.columns:
                labels[drug] = normalize_ast_value(dataset.y.iloc[idx][drug])
            else:
                try:
                    labels[drug] = int(dataset.to_numpy(drug)[idx])
                except Exception:
                    labels[drug] = None
        by_code[str(code)] = labels
    return by_code


def background_signature(labels: dict[str, int | None], focal_drug: str, drugs: list[str]) -> str:
    parts = []
    for drug in drugs:
        if drug == focal_drug:
            continue
        label = labels.get(drug)
        parts.append(f"{drug}={LABEL_CHAR.get(label, UNKNOWN_CHAR)}")
    return "|".join(parts) if parts else "NO_BACKGROUND_DRUGS"


def class_one_probability(estimator, x_test) -> np.ndarray:
    proba = estimator.predict_proba(x_test)
    classes = list(getattr(estimator, "classes_", []))
    if 1 in classes:
        return proba[:, classes.index(1)]
    if proba.shape[1] == 2:
        return proba[:, 1]
    raise ValueError(f"Cannot find resistant class probability from classes={classes}")


def metric_summary(y_true, probs, threshold=0.5) -> dict:
    from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score

    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    pred = (probs >= threshold).astype(int)
    out = {"accuracy": float(accuracy_score(y_true, pred))}
    if len(np.unique(y_true)) == 2:
        out["auroc"] = float(roc_auc_score(y_true, probs))
        out["auprc"] = float(average_precision_score(y_true, probs))
    else:
        out["auroc"] = None
        out["auprc"] = None
    return out


def fit_fallback_estimator(model_name: str, x_train, y_train, seed: int):
    model_name = str(model_name).lower()
    if model_name in {"lr", "logreg", "logistic", "logistic-regression"}:
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=2000,
                solver="lbfgs",
                random_state=seed,
            ),
        ).fit(x_train, y_train)
    if model_name in {"lightgbm", "lgbm"}:
        from lightgbm import LGBMClassifier
        return LGBMClassifier(
            n_estimators=300,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.9,
            colsample_bytree=0.9,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1,
            verbosity=-1,
        ).fit(x_train, y_train)
    raise ValueError(
        f"Unsupported fallback model={model_name!r}. Use lr or lightgbm, "
        "or rely on the Weis repo exposing run_experiment for this model."
    )


def fit_predict_with_weis_or_fallback(models, x_train, y_train, x_test, y_test, args):
    if str(args.model).lower() in {"lr", "logreg", "logistic", "logistic-regression", "lightgbm", "lgbm"}:
        estimator = fit_fallback_estimator(args.model, x_train, y_train, args.seed)
        probs = class_one_probability(estimator, x_test)
        return metric_summary(y_test, probs), estimator, probs, "sklearn_fallback"

    if hasattr(models, "run_experiment"):
        try:
            results, estimator = models.run_experiment(
                x_train,
                y_train,
                x_test,
                y_test,
                args.model,
                args.n_folds,
                random_state=args.seed,
                verbose=True,
                return_best_estimator=True,
            )
            probs = class_one_probability(estimator, x_test)
            return results, estimator, probs, "weis_run_experiment"
        except TypeError as exc:
            print(f"Weis run_experiment API did not accept return_best_estimator; using fallback estimator. Details: {exc}")
        except Exception as exc:
            print(f"Weis run_experiment failed; using fallback estimator. Details: {type(exc).__name__}: {exc}")
    estimator = fit_fallback_estimator(args.model, x_train, y_train, args.seed)
    probs = class_one_probability(estimator, x_test)
    return metric_summary(y_test, probs), estimator, probs, "sklearn_fallback"


PREDICTION_FIELDS = [
    "isolate_id",
    "site",
    "year",
    "organism",
    "drug",
    "label",
    "prob",
    "background_signature",
    "model_name",
]


def write_csv(path: pathlib.Path, rows: list[dict], fields: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def write_partial_outputs(output_dir: pathlib.Path, prediction_rows: list[dict], raw_results: list[dict]) -> None:
    write_csv(output_dir / "weis_predictions_long.partial.csv", prediction_rows, PREDICTION_FIELDS)
    (output_dir / "weis_raw_results.partial.json").write_text(json.dumps(raw_results, indent=2) + "\n")


def run(args: argparse.Namespace) -> None:
    models = load_weis_modules(args.weis_repo)
    os.environ["DRIAMS_ROOT"] = str(args.driams_root)

    from maldi_learn.driams import DRIAMSDatasetExplorer

    drugs = split_csv_arg(args.drugs)
    test_sites = split_csv_arg(args.test_sites)
    output_dir = args.output_dir
    output_dir.mkdir(parents=True, exist_ok=True)

    explorer = DRIAMSDatasetExplorer(str(args.driams_root))
    train_years = explorer.available_years(args.train_site)
    raw_results = []
    prediction_rows: list[dict] = []

    for test_site in test_sites:
        test_years = explorer.available_years(test_site)
        background_map = load_background_label_map(
            driams_root=str(args.driams_root),
            site=test_site,
            years=test_years,
            species=args.species,
            drugs=drugs,
        )

        for drug in drugs:
            train_dataset, x_train, y_train, train_codes, train_strat_fn = load_dataset_for_focal(
                driams_root=str(args.driams_root),
                site=args.train_site,
                years=train_years,
                species=args.species,
                antibiotic=drug,
            )
            test_dataset, x_test_full, y_test_full, test_codes, test_strat_fn = load_dataset_for_focal(
                driams_root=str(args.driams_root),
                site=test_site,
                years=test_years,
                species=args.species,
                antibiotic=drug,
            )

            # maldi-learn stratification helpers expect the metadata frame, not the
            # already-extracted NumPy label vector. Some site/drug subsets are empty
            # after maldi-learn filtering; skip those instead of killing the run.
            try:
                train_idx, _ = train_strat_fn(train_dataset.y, antibiotic=drug, random_state=args.seed)
                _, test_idx = test_strat_fn(test_dataset.y, antibiotic=drug, random_state=args.seed)
            except ValueError as exc:
                print(f"SKIP {test_site} {drug}: stratification failed: {exc}", flush=True)
                raw_results.append({
                    "train_site": args.train_site,
                    "test_site": test_site,
                    "train_years": train_years,
                    "test_years": test_years,
                    "species": args.species,
                    "drug": drug,
                    "model": args.model,
                    "seed": args.seed,
                    "skipped": True,
                    "skip_reason": f"stratification failed: {exc}",
                })
                write_partial_outputs(output_dir, prediction_rows, raw_results)
                continue

            train_idx = np.asarray(train_idx, dtype=int)
            test_idx = np.asarray(test_idx, dtype=int)
            x_train_used = x_train[train_idx]
            y_train_used = y_train[train_idx]
            x_test = x_test_full[test_idx]
            y_test = y_test_full[test_idx]
            codes_test = [test_codes[idx] for idx in test_idx]

            if len(y_train_used) == 0 or len(y_test) == 0 or len(np.unique(y_train_used)) < 2 or len(np.unique(y_test)) < 2:
                reason = (
                    f"insufficient labels after split: n_train={len(y_train_used)}, "
                    f"train_classes={sorted(set(map(int, y_train_used))) if len(y_train_used) else []}, "
                    f"n_test={len(y_test)}, test_classes={sorted(set(map(int, y_test))) if len(y_test) else []}"
                )
                print(f"SKIP {test_site} {drug}: {reason}", flush=True)
                raw_results.append({
                    "train_site": args.train_site,
                    "test_site": test_site,
                    "train_years": train_years,
                    "test_years": test_years,
                    "species": args.species,
                    "drug": drug,
                    "model": args.model,
                    "seed": args.seed,
                    "skipped": True,
                    "skip_reason": reason,
                })
                write_partial_outputs(output_dir, prediction_rows, raw_results)
                continue

            results, estimator, probs, fit_source = fit_predict_with_weis_or_fallback(
                models, x_train_used, y_train_used, x_test, y_test, args
            )

            raw_results.append(
                {
                    "train_site": args.train_site,
                    "test_site": test_site,
                    "train_years": train_years,
                    "test_years": test_years,
                    "species": args.species,
                    "drug": drug,
                    "model": args.model,
                    "seed": args.seed,
                    "n_train": int(len(y_train_used)),
                    "n_test": int(len(y_test)),
                    "auroc": results.get("auroc"),
                    "auprc": results.get("auprc"),
                    "accuracy": results.get("accuracy"),
                    "best_params": results.get("best_params"),
                    "fit_source": fit_source,
                }
            )

            year = "_".join(str(y) for y in test_years)
            for code, label, prob in zip(codes_test, y_test, probs):
                labels = background_map.get(str(code), {})
                prediction_rows.append(
                    {
                        "isolate_id": str(code),
                        "site": test_site,
                        "year": year,
                        "organism": args.species,
                        "drug": drug,
                        "label": int(label),
                        "prob": float(prob),
                        "background_signature": background_signature(labels, drug, drugs),
                        "model_name": f"Weis-{args.model}",
                    }
                )
            print(f"{test_site} {drug}: exported {len(y_test)} prediction rows", flush=True)
            write_partial_outputs(output_dir, prediction_rows, raw_results)

    predictions_csv = output_dir / "weis_predictions_long.csv"
    write_csv(predictions_csv, prediction_rows, PREDICTION_FIELDS)
    (output_dir / "weis_raw_results.json").write_text(json.dumps(raw_results, indent=2) + "\n")

    audit_dir = output_dir / "audit"
    cmd = [
        sys.executable,
        str(args.audit_script),
        "--predictions-csv",
        str(predictions_csv),
        "--background-signature-col",
        "background_signature",
        "--model-name",
        f"Weis-{args.model}",
        "--output-dir",
        str(audit_dir),
        "--bootstrap-n",
        str(args.bootstrap_n),
        "--permutation-n",
        str(args.permutation_n),
    ]
    if args.match_year:
        cmd.append("--match-year")
    subprocess.run(cmd, check=True)
    print(f"Wrote Weis prediction table to {predictions_csv}")
    print(f"Wrote Weis background audit to {audit_dir}")


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--weis-repo", type=pathlib.Path, required=True)
    parser.add_argument("--driams-root", type=pathlib.Path, required=True)
    parser.add_argument("--audit-script", type=pathlib.Path, default=pathlib.Path("run_background_audit_framework.py"))
    parser.add_argument("--output-dir", type=pathlib.Path, required=True)
    parser.add_argument("--train-site", default="DRIAMS-A")
    parser.add_argument("--test-sites", default="DRIAMS-B,DRIAMS-C,DRIAMS-D")
    parser.add_argument("--species", default="Escherichia coli")
    parser.add_argument(
        "--drugs",
        default="Ciprofloxacin,Norfloxacin,Amoxicillin-Clavulanic acid,Ceftriaxone,Ceftazidime,Cefepime",
    )
    parser.add_argument("--model", default="lr")
    parser.add_argument("--seed", type=int, default=35)
    parser.add_argument("--n-folds", type=int, default=5)
    parser.add_argument("--bootstrap-n", type=int, default=500)
    parser.add_argument("--permutation-n", type=int, default=500)
    parser.add_argument("--match-year", action="store_true")
    return parser


def main() -> None:
    run(build_parser().parse_args())


if __name__ == "__main__":
    main()


Overwriting /kaggle/working/export_weis_predictions_for_audit.py


In [9]:
from pathlib import Path
script = Path("/kaggle/working/export_weis_predictions_for_audit.py")
audit = Path("/kaggle/working/run_background_audit_framework.py")
assert script.exists(), f"Run the %%writefile exporter cell first; missing {script}"
assert audit.exists(), f"Run the %%writefile audit-framework cell first; missing {audit}"

!python -u /kaggle/working/export_weis_predictions_for_audit.py \
  --weis-repo /kaggle/working/maldi_amr \
  --driams-root /kaggle/input/datasets/drscarlat/driams \
  --audit-script /kaggle/working/run_background_audit_framework.py \
  --species "Escherichia coli" \
  --drugs "Ciprofloxacin,Norfloxacin,Amoxicillin-Clavulanic acid,Ceftriaxone,Ceftazidime,Cefepime" \
  --model lr \
  --seed 35 \
  --bootstrap-n 200 \
  --permutation-n 200 \
  --output-dir /kaggle/working/weis_background_audit_lr


/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_encoded[valid_columns] = y_encoded[valid_columns].replace(
/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_encoded[valid_columns] = y_encoded[valid_columns].replace(
/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be remo

In [ ]:
from pathlib import Path
script = Path("/kaggle/working/export_weis_predictions_for_audit.py")
audit = Path("/kaggle/working/run_background_audit_framework.py")
assert script.exists(), f"Run the %%writefile exporter cell first; missing {script}"
assert audit.exists(), f"Run the %%writefile audit-framework cell first; missing {audit}"

!python -u /kaggle/working/export_weis_predictions_for_audit.py \
  --weis-repo /kaggle/working/maldi_amr \
  --driams-root /kaggle/input/datasets/drscarlat/driams \
  --audit-script /kaggle/working/run_background_audit_framework.py \
  --species "Escherichia coli" \
  --drugs "Ciprofloxacin,Norfloxacin,Amoxicillin-Clavulanic acid,Ceftriaxone,Ceftazidime,Cefepime" \
  --model lightgbm \
  --seed 35 \
  --bootstrap-n 200 \
  --permutation-n 200 \
  --output-dir /kaggle/working/weis_background_audit_lightgbm


/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_encoded[valid_columns] = y_encoded[valid_columns].replace(
/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_encoded[valid_columns] = y_encoded[valid_columns].replace(
/usr/local/lib/python3.12/dist-packages/maldi_learn/preprocessing/generic.py:73: FutureWarning: Downcasting behavior in `replace` is deprecated and will be remo

In [ ]:
from pathlib import Path
import subprocess
import sys

WORK = Path("/kaggle/working")
EXPORTER = WORK / "export_weis_predictions_for_audit.py"
AUDIT = WORK / "run_background_audit_framework.py"
WEIS_REPO = WORK / "maldi_amr"
DRIAMS_ROOT = Path("/kaggle/input/datasets/drscarlat/driams")
DRUGS = "Ciprofloxacin,Norfloxacin,Amoxicillin-Clavulanic acid,Ceftriaxone,Ceftazidime,Cefepime"
SPECIES = "Escherichia coli"
SEED = "35"
BOOTSTRAP_N = "200"
PERMUTATION_N = "200"

assert EXPORTER.exists(), f"Run the exporter %%writefile cell first; missing {EXPORTER}"
assert AUDIT.exists(), f"Run the audit-framework %%writefile cell first; missing {AUDIT}"
assert WEIS_REPO.exists(), f"Run the git clone cell first; missing {WEIS_REPO}"
assert DRIAMS_ROOT.exists(), f"Attach DRIAMS dataset; missing {DRIAMS_ROOT}"

def run_export(model_name: str, out_dir: Path):
    final_predictions = out_dir / "weis_predictions_long.csv"
    final_summary = out_dir / "audit" / "background_matched_audit_summary.csv"
    if final_predictions.exists() and final_summary.exists():
        print(f"SKIP Weis-{model_name}: final outputs already exist at {out_dir}")
        return
    cmd = [
        sys.executable, "-u", str(EXPORTER),
        "--weis-repo", str(WEIS_REPO),
        "--driams-root", str(DRIAMS_ROOT),
        "--audit-script", str(AUDIT),
        "--species", SPECIES,
        "--drugs", DRUGS,
        "--model", model_name,
        "--seed", SEED,
        "--bootstrap-n", BOOTSTRAP_N,
        "--permutation-n", PERMUTATION_N,
        "--output-dir", str(out_dir),
    ]
    print("\n=== Running", model_name, "===")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

def run_match_year(model_name: str, out_dir: Path):
    predictions = out_dir / "weis_predictions_long.csv"
    match_dir = out_dir / "audit_match_year"
    match_summary = match_dir / "background_matched_audit_summary.csv"
    if not predictions.exists():
        print(f"SKIP match-year Weis-{model_name}: missing {predictions}")
        return
    if match_summary.exists():
        print(f"SKIP match-year Weis-{model_name}: final output already exists at {match_summary}")
        return
    cmd = [
        sys.executable, "-u", str(AUDIT),
        "--predictions-csv", str(predictions),
        "--background-signature-col", "background_signature",
        "--model-name", f"Weis-{model_name}",
        "--output-dir", str(match_dir),
        "--bootstrap-n", BOOTSTRAP_N,
        "--permutation-n", PERMUTATION_N,
        "--match-year",
    ]
    print("\n=== Running match-year sensitivity for", model_name, "===")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

LR_DIR = WORK / "weis_background_audit_lr"
LGBM_DIR = WORK / "weis_background_audit_lightgbm"
run_export("lr", LR_DIR)
run_export("lightgbm", LGBM_DIR)
run_match_year("lr", LR_DIR)
run_match_year("lightgbm", LGBM_DIR)

print("\nFull Weis suite outputs:")
for p in [LR_DIR, LGBM_DIR]:
    print(" ", p)
    for child in [
        p / "weis_predictions_long.csv",
        p / "audit" / "background_matched_audit_summary.csv",
        p / "audit" / "background_audit_with_resistance_ecology.csv",
        p / "audit_match_year" / "background_matched_audit_summary.csv",
    ]:
        print("   ", "OK" if child.exists() else "MISSING", child)


In [ ]:
from pathlib import Path
import math
import shutil
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WORK = Path("/kaggle/working")
ART = WORK / "weis_full_analysis_artifacts"
ART.mkdir(parents=True, exist_ok=True)

MODEL_DIRS = {
    "Weis LR": WORK / "weis_background_audit_lr",
    "Weis LightGBM": WORK / "weis_background_audit_lightgbm",
}
ECOLOGY_BLOCKS = {
    "quinolone block": {"Ciprofloxacin", "Norfloxacin"},
    "beta-lactam/cephalosporin block": {"Amoxicillin-Clavulanic acid", "Ceftriaxone", "Ceftazidime", "Cefepime"},
}

def ecology_block(drug):
    for block, drugs in ECOLOGY_BLOCKS.items():
        if drug in drugs:
            return block
    return "other"

def signal_retention(centered_auc, raw_auc):
    if pd.isna(centered_auc) or pd.isna(raw_auc) or abs(raw_auc - 0.5) < 1e-9:
        return np.nan
    return (centered_auc - 0.5) / (raw_auc - 0.5)

def read_summary(path, model, match_year=False):
    if not path.exists():
        print(f"Missing summary for {model}: {path}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["model"] = model
    df["analysis"] = "match_year" if match_year else "primary"
    df["ecology_block"] = df["drug"].map(ecology_block)
    df["background_centered_auc"] = df["stratum_centered_auc"]
    df["centered_signal_retention"] = [signal_retention(c, r) for c, r in zip(df["background_centered_auc"], df["raw_auc"])]
    if "pairwise_within_background_accuracy" not in df.columns and "pairwise_accuracy" in df.columns:
        df["pairwise_within_background_accuracy"] = df["pairwise_accuracy"]
    return df

frames = []
for model, base in MODEL_DIRS.items():
    frames.append(read_summary(base / "audit" / "background_matched_audit_summary.csv", model, False))
    frames.append(read_summary(base / "audit_match_year" / "background_matched_audit_summary.csv", model, True))
all_summary = pd.concat([f for f in frames if not f.empty], ignore_index=True)
if all_summary.empty:
    raise FileNotFoundError("No Weis audit summaries found. Run the full suite cell first.")

all_summary["interpretable"] = all_summary["adequacy_label"].astype(str).eq("interpretable")
primary = all_summary[all_summary["analysis"] == "primary"].copy()
primary_core = primary[primary["interpretable"]].copy()

main_cols = [
    "model", "analysis", "site", "drug", "ecology_block", "raw_auc", "background_centered_auc",
    "centered_signal_retention", "matched_auc", "matched_retention", "pairwise_within_background_accuracy",
    "pairwise_comparisons", "adequacy_label", "interpretation_category", "n_total", "n_matched",
]
all_summary[[c for c in main_cols if c in all_summary.columns]].to_csv(ART / "weis_full_primary_and_match_year_summary.csv", index=False)
primary[[c for c in main_cols if c in primary.columns]].to_csv(ART / "weis_primary_model_comparison_table.csv", index=False)
primary_core.to_csv(ART / "weis_primary_interpretable_rows.csv", index=False)
primary[~primary["interpretable"]].to_csv(ART / "weis_primary_noninterpretable_rows.csv", index=False)

block_summary = (primary_core.groupby(["model", "ecology_block"], dropna=False)
    .agg(
        n=("background_centered_auc", "count"),
        raw_mean=("raw_auc", "mean"),
        centered_mean=("background_centered_auc", "mean"),
        retention_mean=("centered_signal_retention", "mean"),
        median_pairwise=("pairwise_comparisons", "median"),
    )
    .reset_index())
block_summary.to_csv(ART / "weis_ecology_block_summary.csv", index=False)

pivot = primary.pivot_table(
    index=["site", "organism", "drug", "ecology_block"],
    columns="model",
    values=["raw_auc", "background_centered_auc", "matched_retention"],
    aggfunc="first",
)
pivot.columns = ["__".join(col).strip() for col in pivot.columns.to_flat_index()]
pivot = pivot.reset_index()
pivot.to_csv(ART / "weis_lr_vs_lightgbm_wide_comparison.csv", index=False)

match_year = all_summary[all_summary["analysis"] == "match_year"].copy()
if not match_year.empty:
    my = primary[["model", "site", "organism", "drug", "background_centered_auc", "matched_retention"]].merge(
        match_year[["model", "site", "organism", "drug", "background_centered_auc", "matched_retention"]],
        on=["model", "site", "organism", "drug"], how="left", suffixes=("", "_match_year"))
    my["delta_centered_auc_match_year"] = my["background_centered_auc_match_year"] - my["background_centered_auc"]
    my.to_csv(ART / "weis_match_year_sensitivity_decision.csv", index=False)
else:
    my = pd.DataFrame()

# Main figure: raw AUC to background-centered AUC by model and ecology block.
plot_df = primary.dropna(subset=["raw_auc", "background_centered_auc"]).sort_values(["model", "ecology_block", "drug", "site"])
models = list(MODEL_DIRS.keys())
colors = {"quinolone block": "#1677b7", "beta-lactam/cephalosporin block": "#c44e52", "other": "#555555"}
fig, axes = plt.subplots(1, len(models), figsize=(6.2 * len(models), max(6, 0.32 * len(plot_df) / max(1, len(models)))), sharey=True)
if len(models) == 1:
    axes = [axes]
for ax, model in zip(axes, models):
    sub = plot_df[plot_df["model"] == model].reset_index(drop=True)
    y = np.arange(len(sub))
    for idx, row in sub.iterrows():
        color = colors.get(row["ecology_block"], "#555555")
        ax.plot([row["raw_auc"], row["background_centered_auc"]], [idx, idx], color=color, alpha=0.75, linewidth=2)
        ax.scatter(row["raw_auc"], idx, color=color, marker="o", s=28)
        ax.scatter(row["background_centered_auc"], idx, color=color, marker="s", s=28)
    labels = [f"{r.site} | {r.drug}" for r in sub.itertuples()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels if ax is axes[0] else [], fontsize=8)
    ax.axvline(0.5, color="#888888", linestyle="--", linewidth=1)
    ax.set_xlim(0.25, 0.95)
    ax.set_xlabel("AUC")
    ax.set_title(model)
    ax.grid(axis="x", alpha=0.2)
axes[0].set_ylabel("site | drug")
handles = [plt.Line2D([0], [0], color=c, lw=3, label=k) for k, c in colors.items() if k in set(plot_df["ecology_block"])]
marker_handles = [
    plt.Line2D([0], [0], marker="o", color="black", lw=0, label="raw AUC"),
    plt.Line2D([0], [0], marker="s", color="black", lw=0, label="background-centered AUC"),
]
fig.legend(handles=handles + marker_handles, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Weis-style baselines: raw external AUC to background-centered AUC", y=0.98)
fig.tight_layout(rect=[0, 0.08, 1, 0.94])
fig.savefig(ART / "weis_raw_to_background_centered_auc.png", dpi=220)
fig.savefig(ART / "weis_raw_to_background_centered_auc.pdf")
plt.show()

# Ecology/network inventories from each model audit.
for model, base in MODEL_DIRS.items():
    safe = model.lower().replace(" ", "_")
    for rel in [
        "audit/background_audit_with_resistance_ecology.csv",
        "audit/cross_resistance_edges.csv",
        "audit/cross_resistance_prevalence.csv",
        "audit/background_audit_report.md",
    ]:
        src = base / rel
        if src.exists():
            shutil.copy2(src, ART / f"{safe}_{Path(rel).name}")

# Short paper-ready paragraph.
def fmt(x):
    return "NA" if pd.isna(x) else f"{x:.2f}"

lr_blocks = block_summary[block_summary["model"] == "Weis LR"].set_index("ecology_block")
lgbm_blocks = block_summary[block_summary["model"] == "Weis LightGBM"].set_index("ecology_block")
parts = []
for model_name, blocks in [("Weis LR", lr_blocks), ("Weis LightGBM", lgbm_blocks)]:
    quin = blocks.loc["quinolone block"] if "quinolone block" in blocks.index else None
    beta = blocks.loc["beta-lactam/cephalosporin block"] if "beta-lactam/cephalosporin block" in blocks.index else None
    parts.append(
        f"For {model_name}, interpretable quinolone rows had mean raw AUC {fmt(None if quin is None else quin.raw_mean)} "
        f"and mean background-centered AUC {fmt(None if quin is None else quin.centered_mean)}, while interpretable "
        f"beta-lactam/cephalosporin rows had mean raw AUC {fmt(None if beta is None else beta.raw_mean)} and mean "
        f"background-centered AUC {fmt(None if beta is None else beta.centered_mean)}."
    )
if not my.empty:
    delta = my.dropna(subset=["delta_centered_auc_match_year"])["delta_centered_auc_match_year"]
    sensitivity_sentence = (
        f"The stricter year-matched sensitivity changed background-centered AUC by a median of {delta.median():+.2f} "
        "among rows with valid year-matched estimates, so it is best reported as a robustness check alongside the primary background-matched audit."
        if len(delta) else
        "The stricter year-matched sensitivity produced too few valid estimates to replace the primary background-matched audit."
    )
else:
    sensitivity_sentence = "The year-matched sensitivity was not available for the Weis baselines."

paragraph = (
    "The expanded Weis-style baseline analysis applies the same background-matched transfer audit to logistic-regression "
    "and LightGBM models. " + " ".join(parts) + " " + sensitivity_sentence + " Together, these outputs test whether the two-block ecology pattern is model-family-specific or also appears in classical MALDI-AMR baselines."
)
(ART / "weis_core_results_paragraph.txt").write_text(paragraph + "\n", encoding="utf-8")
(ART / "weis_core_results_paragraph.md").write_text(paragraph + "\n", encoding="utf-8")

# Zip for easy Kaggle download.
zip_path = shutil.make_archive(str(ART), "zip", ART)

print("Main Weis comparison table:")
display(primary[[c for c in main_cols if c in primary.columns]])
print("\nEcology block summary:")
display(block_summary)
if not my.empty:
    print("\nMatch-year sensitivity decision table:")
    display(my)
print("\nCore Results paragraph:\n")
print(textwrap.fill(paragraph, width=110))
print("\nWrote expanded Weis artifacts to", ART)
print("Zipped artifacts:", zip_path)
